# 03 Feature Engineering

Build leakage-safe features and inspect the exported feature columns used by each model.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve().parents[1]
TRAINING = ROOT / 'training'
DATA = ROOT / 'datasets'
sys.path.insert(0, str(TRAINING))

from preprocessing import TrafficPreprocessor, TelemetryPreprocessor, IncidentPreprocessor, canonical_link_id_map, load_capacities, load_physical_baselines, load_universe_config
from split import tick_split
from features import CONGESTION_FEATURE_COLS, TRUST_FEATURE_COLS, TARGETING_FEATURE_COLS, build_congestion_features, build_trust_features, build_targeting_features

In [ ]:
cfg = load_universe_config(DATA / 'universe-config.json')
baselines = load_physical_baselines(cfg)
capacities = load_capacities(cfg)
link_id_map = canonical_link_id_map(cfg)

traffic = pd.read_csv(DATA / 'link_traffic_history.csv').sort_values(['link_id', 'tick'])
traffic_split = tick_split(traffic)
traffic_prep = TrafficPreprocessor().fit(traffic_split.train, baselines, capacities, link_id_map)
traffic_train_features = build_congestion_features(traffic_prep.transform(traffic_split.train))
traffic_val_features = build_congestion_features(traffic_prep.transform(traffic_split.val))
traffic_test_features = build_congestion_features(traffic_prep.transform(traffic_split.test))

display(traffic_train_features[CONGESTION_FEATURE_COLS].head())

In [ ]:
telemetry = pd.read_csv(DATA / 'link_telemetry.csv').sort_values(['link_id', 'tick'])
telemetry_split = tick_split(telemetry)
telemetry_prep = TelemetryPreprocessor().fit(telemetry_split.train, baselines, link_id_map)
telemetry_train_features = build_trust_features(telemetry_prep.transform(telemetry_split.train))
telemetry_val_features = build_trust_features(telemetry_prep.transform(telemetry_split.val))
telemetry_test_features = build_trust_features(telemetry_prep.transform(telemetry_split.test))

incidents = pd.read_csv(DATA / 'link_incident_history.csv').sort_values(['link_id', 'tick'])
incident_split = tick_split(incidents)
incident_prep = IncidentPreprocessor().fit(incident_split.train, link_id_map)
targeting_train_features = build_targeting_features(incident_prep.transform(incident_split.train))
targeting_val_features = build_targeting_features(incident_prep.transform(incident_split.val))
targeting_test_features = build_targeting_features(incident_prep.transform(incident_split.test))

display(telemetry_train_features[TRUST_FEATURE_COLS].head())
display(targeting_train_features[TARGETING_FEATURE_COLS].head())

In [ ]:
pd.DataFrame({
    'model': ['congestion', 'trust', 'targeting'],
    'feature_count': [len(CONGESTION_FEATURE_COLS), len(TRUST_FEATURE_COLS), len(TARGETING_FEATURE_COLS)],
    'features': [CONGESTION_FEATURE_COLS, TRUST_FEATURE_COLS, TARGETING_FEATURE_COLS],
})